# Tutorial 2 – Free vibration of a 1-DoF system (damped)
**Mechanical Vibrations – Centrale Nantes**

Animations of the two damped systems of Tutorial 2:
* **Animation 1** – simple pendulum in air (group A),
* **Animation 2** – vertical mass–spring–damper (group B),
* **Animation 3** – the three types of motion (same system, three damping values).

**How to use it:** run all the cells (*Run → Run all cells*). In the interactive figures, move the **sliders** to change the parameters and use the **Play/Pause** buttons below the figure.
The code cells are hidden (click on them to show the code).

In [ ]:
#@title Setup (run first): interactive figures with sliders
import sys, subprocess
if "pyodide" in sys.modules:                     # JupyterLite (in the browser)
    import piplite
    await piplite.install(["ipympl", "ipywidgets"])
elif "google.colab" in sys.modules:              # Google Colab
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ipympl"])
    from google.colab import output
    output.enable_custom_widget_manager()
try:
    import ipympl
    get_ipython().run_line_magic("matplotlib", "widget")
except Exception:
    print("ipympl is not installed: the animations will be shown without sliders (pip install ipympl).")

In [ ]:
#@title Helper functions (run this cell first)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

plt.rcParams.update({"figure.dpi": 90, "animation.embed_limit": 60, "font.size": 10,
                     # LaTeX-like rendering without a LaTeX installation: $...$ formulas and text in STIX (Times-like)
                     "mathtext.fontset": "stix", "font.family": "STIXGeneral"})
BLUE, RED, GREEN, GREY = "#1f4e99", "#a3201d", "#2e6b30", "#777777"


def rk4(f, y0, t):
    """Fixed-step Runge-Kutta 4 integration of dy/dt = f(t, y)."""
    y = np.zeros((len(t), len(y0)))
    y[0] = y0
    for i in range(len(t) - 1):
        h = t[i + 1] - t[i]
        k1 = f(t[i], y[i])
        k2 = f(t[i] + h / 2, y[i] + h / 2 * k1)
        k3 = f(t[i] + h / 2, y[i] + h / 2 * k2)
        k4 = f(t[i] + h, y[i] + h * k3)
        y[i + 1] = y[i] + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
    return y


def spring_xy(p0, p1, n=7, width=0.012, lead=0.15):
    """Zig-zag polyline of a spring between points p0 and p1."""
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    L = np.linalg.norm(p1 - p0)
    u = (p1 - p0) / L
    v = np.array([-u[1], u[0]])
    pts = [p0, p0 + lead * L * u]
    for i in range(2 * n):
        s = lead * L + (1 - 2 * lead) * L * (i + 0.5) / (2 * n)
        pts.append(p0 + s * u + (width if i % 2 == 0 else -width) * v)
    pts += [p1 - lead * L * u, p1]
    pts = np.array(pts)
    return pts[:, 0], pts[:, 1]


def hatch(ax, x0, x1, y, down=True, n=12, size=0.012):
    """Draw a fixed support (line + hatching)."""
    ax.plot([x0, x1], [y, y], color="k", lw=2)
    s = -size if down else size
    for x in np.linspace(x0, x1, n):
        ax.plot([x, x + size], [y, y - s], color="k", lw=0.8)


def make_anim(fig, update, nframes, fps=25):
    """Build an animation and remember what is needed to export it."""
    anim = animation.FuncAnimation(fig, update, frames=nframes, interval=1000 / fps, blit=False)
    anim.update_func, anim.nframes = update, nframes
    return anim


def show(anim):
    """Display an animation in the notebook (works in Jupyter, JupyterLite and Colab)."""
    plt.close(anim._fig)
    display(HTML(anim.to_jshtml()))


def export_frames(anim, name, folder="gifs", dpi=120):
    """Teacher only: save the frames as name-0.png, name-1.png, ... for the LaTeX animate package."""
    import os
    os.makedirs(f"{folder}/{name}", exist_ok=True)
    for i in range(anim.nframes):
        anim.update_func(i)
        anim._fig.savefig(f"{folder}/{name}/{name}-{i}.png", dpi=dpi)
    print(f"{anim.nframes} frames saved in {folder}/{name}/ (last index {anim.nframes - 1})")


# ---------------------------------------------------------------------------
# Live animation: sliders drawn on the figure + Play/Pause widget below it.
# Needs ipympl (Jupyter, JupyterLite, VS Code, Colab). The time steps are driven
# by the ipywidgets "Play" widget (timer in the browser), which also works in
# JupyterLite where matplotlib timers do not run.
# Without ipympl, the animation is shown with the default parameters.
# ---------------------------------------------------------------------------
from matplotlib.widgets import Slider

INTERACTIVE = "ipympl" in plt.get_backend().lower() or "widget" in plt.get_backend().lower()


class LiveAnimation:
    """
    sliders : list of (name, label, vmin, vmax, vinit, step)
    compute : function(params dict) -> data dict (solution on a fine time grid)
    setup   : function(data) -> None, updates the static elements (curves, limits...)
    frame   : function(data, t) -> None, updates the moving elements at time t
    T       : simulated duration (s), played in a loop
              (compute may return data["T"] to change it, e.g. a whole number of periods)
    slow    : slow motion factor
    """

    def __init__(self, fig, sliders, compute, setup, frame, T, slow=1, fps=20):
        self.fig, self.compute, self.setup, self.frame = fig, compute, setup, frame
        self.T, self.slow, self.fps = T, slow, fps
        n = len(sliders)
        fig.subplots_adjust(bottom=0.08 + 0.045 * n)
        self.sliders = {}
        for j, (name, label, vmin, vmax, vinit, step) in enumerate(sliders):
            ax = fig.add_axes([0.22, 0.02 + 0.045 * (n - 1 - j), 0.5, 0.025])
            s = Slider(ax, label, vmin, vmax, valinit=vinit, valstep=step)
            s.on_changed(self._changed)
            self.sliders[name] = s
        for a in fig.axes:                        # drawings with equal aspect: keep them aligned at the top
            if a.get_aspect() == 1.0:
                a.set_anchor("N")
        self.i = 0
        self.data = self.compute(self.params())
        self._loop()
        self.setup(self.data)
        self.frame(self.data, 0.0)
        if INTERACTIVE:
            import ipywidgets as W
            fig.canvas.header_visible = False
            self.play = W.Play(value=0, min=0, max=self.nframes - 1, step=1,
                               interval=int(1000 / fps), repeat=True, description="Play")
            self.play.observe(self._tick, names="value")
            plt.show()                            # figure first, controls below
            display(W.HBox([self.play, W.Label("play / pause / stop (loop button: repeat)")]))
            try:
                self.play.playing = True          # start automatically
            except Exception:
                pass
        else:
            print("Static version (default parameters): install ipympl to use the sliders.")
            show(make_anim(fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop), self.nframes, fps))

    def params(self):
        return {name: s.val for name, s in self.sliders.items()}

    def _loop(self):
        """Loop duration (from compute if given) and number of frames of one loop."""
        self.Tloop = self.data.get("T", self.T)
        self.nframes = max(2, int(round(self.Tloop * self.fps * self.slow)))

    def _t(self):
        return (self.i % self.nframes) / self.nframes * self.Tloop

    def _changed(self, _):
        self.data = self.compute(self.params())
        self._loop()
        if hasattr(self, "play"):
            self.i = min(self.i, self.nframes - 1)
            self.play.max = self.nframes - 1
        self.setup(self.data)
        self.frame(self.data, self._t())
        self.fig.canvas.draw_idle()

    def _tick(self, change):
        self.i = change["new"]
        self.frame(self.data, self._t())
        self.fig.canvas.draw_idle()

    def export(self, name, folder="gifs", dpi=120):
        """Teacher only: frames for the LaTeX animate package (current slider values)."""
        export_frames(make_anim(self.fig, lambda i: self.frame(self.data, i / self.nframes * self.Tloop),
                                self.nframes, self.fps), name, folder, dpi)


def period(t, q):
    """Period measured from the upward zero crossings of q(t)."""
    idx = np.where((q[:-1] < 0) & (q[1:] >= 0))[0]
    return np.mean(np.diff(t[idx])) if len(idx) > 1 else np.nan


class Dashpot:
    """Viscous damper drawn between p0 (cylinder side) and p1 (piston side).
    The geometry is fixed by the first call to set() (or after reset()): the piston then moves inside the cylinder."""

    def __init__(self, ax, width=0.01, **kw):
        kw.setdefault("color", "k"); kw.setdefault("lw", 1.2)
        self.w, self.geom = width, None
        self.lines = [ax.plot([], [], **kw)[0] for _ in range(3)]

    def reset(self):
        self.geom = None

    def set(self, p0, p1):
        p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
        L = np.linalg.norm(p1 - p0); u = (p1 - p0) / L; v = np.array([-u[1], u[0]])
        if self.geom is None:
            self.geom = (0.1 * L, 0.65 * L, 0.6 * L)          # stem, cylinder length, piston rod length
        a, cyl, rod = self.geom
        P = lambda s, d=0.0: p0 + s * u + d * v
        stem = np.array([p0, P(a)])
        cylinder = np.array([P(a + cyl, self.w), P(a, self.w), P(a, -self.w), P(a + cyl, -self.w)])
        sp = L - rod
        piston = np.array([P(sp, 0.8 * self.w), P(sp, -0.8 * self.w), P(sp), p1])
        for line, pts in zip(self.lines, [stem, cylinder, piston]):
            line.set_data(pts[:, 0], pts[:, 1])


def double_pendulum_rhs(g, L):
    """Exact equations of a double pendulum with two equal masses and two equal lengths (absolute angles)."""
    def f(_, s):
        t1, t2, w1, w2 = s
        d = t1 - t2
        den = L * (3 - np.cos(2 * d))
        a1 = (-3 * g * np.sin(t1) - g * np.sin(t1 - 2 * t2) - 2 * np.sin(d) * (w2 ** 2 * L + w1 ** 2 * L * np.cos(d))) / den
        a2 = (2 * np.sin(d) * (2 * w1 ** 2 * L + 2 * g * np.cos(t1) + w2 ** 2 * L * np.cos(d))) / den
        return np.array([w1, w2, a1, a2])
    return f


def draw_two_masses(ax, x1, x2, objs=None, gap=0.3, w=0.12, h=0.1, scale=1.0):
    """Two masses between two walls (walls at 0 and 3*gap+2*w); x1, x2 in the units of the drawing."""
    xa = gap + w / 2 + scale * x1
    xb = 2 * gap + 1.5 * w + scale * x2
    xr = 3 * gap + 2 * w
    if objs is None:
        hatch_v = lambda x, s: [ax.plot([x, x], [-0.02, h + 0.06], "k", lw=2)] + \
            [ax.plot([x, x + s], [y, y + 0.02], "k", lw=0.7) for y in np.linspace(-0.02, h + 0.04, 7)]
        hatch_v(0, -0.025); hatch_v(xr, 0.025)
        ax.plot([0, xr], [-0.02, -0.02], "k", lw=1)
        sps = [ax.plot([], [], "k", lw=1.1)[0] for _ in range(3)]
        boxes = [plt.Rectangle((0, 0), w, h, fc="#f4c7c3", ec="k") for _ in range(2)]
        for b in boxes:
            ax.add_patch(b)
        objs = (sps, boxes)
    sps, boxes = objs
    yc = h / 2
    ends = [(0, xa - w / 2), (xa + w / 2, xb - w / 2), (xb + w / 2, xr)]
    for s, (p, q) in zip(sps, ends):
        s.set_data(*spring_xy((p, yc), (q, yc), n=6, width=0.025))
    boxes[0].set_x(xa - w / 2); boxes[1].set_x(xb - w / 2)
    return objs


## Animation 1 – Simple pendulum in air (group A)
The pendulum is released from $\theta_0$ without initial velocity. The air friction is modelled by a viscous force $-c\,\vec v$ on the mass. The dashed curves show the envelope of the motion.

**Observe:** how the decay of the oscillations changes with $c$, $m$ and $L$. Does the period change much?

In [ ]:
#@title Animation 1
def animation_1():
    g, Tsim = 9.81, 20.0

    def compute(p):
        L, m, c, th0 = p["L"], p["m"], p["c"], np.radians(p["th0"])
        tf = np.linspace(0, Tsim, 8001)
        th = rk4(lambda _, s: np.array([s[1], -g / L * np.sin(s[0]) - c / m * s[1]]), [th0, 0.0], tf)[:, 0]
        pk = np.r_[False, (th[1:-1] > th[:-2]) & (th[1:-1] >= th[2:]), False]      # maxima of the motion
        return dict(tf=tf, th=th, L=L, m=m, tpk=np.r_[0.0, tf[pk]], pk=np.r_[th0, th[pk]])

    fig = plt.figure(figsize=(9, 6.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 2], top=0.93, wspace=0.25)
    ax, axq = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax.set_aspect("equal"); ax.axis("off"); ax.set_xlim(-0.6, 0.6); ax.set_ylim(-0.85, 0.05)
    hatch(ax, -0.12, 0.12, 0.0, down=False, size=0.03)
    vert, = ax.plot([0, 0], [0, -1], color=GREY, ls="--", lw=0.8)
    rod, = ax.plot([], [], "k", lw=1.5)
    bob, = ax.plot([], [], "o", color=RED)
    ax.set_title("simple pendulum in air")
    curve, = axq.plot([], [], color=RED, lw=1.5)
    env1, = axq.plot([], [], "k--", lw=1)
    env2, = axq.plot([], [], "k--", lw=1)
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlim(0, Tsim); axq.set_xlabel(r"$t$ (s)"); axq.set_ylabel(r"$\theta$ (°)"); axq.grid(alpha=0.3)

    def setup(d):
        vert.set_ydata([0, -d["L"]]); bob.set_markersize(8 + 10 * d["m"] ** (1 / 3))
        curve.set_data(d["tf"], np.degrees(d["th"]))
        env1.set_data(d["tpk"], np.degrees(d["pk"])); env2.set_data(d["tpk"], -np.degrees(d["pk"]))
        a = np.degrees(np.max(np.abs(d["th"]))); axq.set_ylim(-1.2 * a, 1.2 * a)

    def frame(d, t):
        th = np.interp(t, d["tf"], d["th"])
        x, y = d["L"] * np.sin(th), -d["L"] * np.cos(th)
        rod.set_data([0, x], [0, y]); bob.set_data([x], [y]); cursor.set_xdata([t, t])

    live = LiveAnimation(fig, [("L", r"$L$ (m)", 0.1, 0.8, 0.25, 0.05), ("m", r"$m$ (kg)", 0.05, 1.0, 0.2, 0.05),
                               ("c", r"$c$ (N s/m)", 0.0, 0.5, 0.05, 0.01), ("th0", r"$\theta_0$ (°)", 1, 45, 10, 1)],
                         compute, setup, frame, Tsim, slow=0.25)
    return live

live1 = animation_1()

## Animation 2 – Vertical mass–spring–damper (group B)
The mass is released from $u_0$ (measured from the equilibrium position) without initial velocity.

**Observe:** how the decay and the type of motion change with $c$. For which value of $c$ does the mass stop oscillating?

In [ ]:
#@title Animation 2
def animation_2():
    g, Tsim = 9.81, 1.5
    L0, hbox = 0.06, 0.02

    def compute(p):
        m, k, c, u0 = p["m"], p["k"], p["c"], p["u0"] / 100
        tf = np.linspace(0, Tsim, 3001)
        u = rk4(lambda _, s: np.array([s[1], -k / m * s[0] - c / m * s[1]]), [u0, 0.0], tf)[:, 0]
        return dict(tf=tf, u=u, ye=m * g / k, m=m, u0=u0)

    fig = plt.figure(figsize=(9, 6.5))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.3, 2], top=0.93, wspace=0.25)
    ax, axq = fig.add_subplot(gs[0]), fig.add_subplot(gs[1])
    ax.set_aspect("equal"); ax.axis("off")
    hatch(ax, -0.05, 0.05, 0.0, down=False, size=0.01)
    eq_line = ax.axhline(0, color=GREY, ls="--", lw=0.8)
    sp, = ax.plot([], [], "k", lw=1.2)
    dp = Dashpot(ax, width=0.008)
    box = plt.Rectangle((-0.03, 0), 0.06, hbox, fc="#f4c7c3", ec="k"); ax.add_patch(box)
    ax.set_title("mass–spring–damper")
    curve, = axq.plot([], [], color=BLUE, lw=2)
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlim(0, Tsim); axq.set_xlabel(r"$t$ (s)"); axq.set_ylabel(r"$u$ (cm)"); axq.grid(alpha=0.3)

    def setup(d):
        a = max(abs(d["u0"]), 1e-3)
        ax.set_xlim(-0.07, 0.07); ax.set_ylim(-(L0 + d["ye"] + a + hbox + 0.01), 0.01)
        eq_line.set_ydata([-(L0 + d["ye"])] * 2)
        curve.set_data(d["tf"], 100 * d["u"]); axq.set_ylim(-110 * a, 110 * a)
        dp.reset()

    def frame(d, t):
        y = d["ye"] + np.interp(t, d["tf"], d["u"])
        sp.set_data(*spring_xy((-0.015, 0), (-0.015, -(L0 + y)), width=0.007))
        dp.set((0.015, 0), (0.015, -(L0 + y)))
        box.set_y(-(L0 + y) - hbox); cursor.set_xdata([t, t])

    live = LiveAnimation(fig, [("m", r"$m$ (kg)", 0.1, 1.0, 0.5, 0.05), ("k", r"$k$ (N/m)", 100, 500, 200, 10),
                               ("c", r"$c$ (N s/m)", 0.0, 50.0, 2.0, 0.5), ("u0", r"$u_0$ (cm)", -3, 3, 1, 0.25)],
                         compute, setup, frame, Tsim, slow=2)
    return live

live2 = animation_2()

## Animation 3 – Three types of motion
Three identical mass–spring–damper systems ($m = 0.5$ kg, $k = 200$ N/m) with three different damping coefficients, released from the same initial displacement.

**Observe:** which system oscillates? Which one comes back to equilibrium the fastest without oscillating?

In [ ]:
#@title Animation 3
def animation_3():
    m, k, u0, slow, fps, T3 = 0.5, 200.0, 0.01, 2, 25, 0.8
    cs = [2.0, 20.0, 40.0]                                     # c (N s/m)
    cols, names = [BLUE, RED, GREEN], [r"$c$ = 2 N s/m", r"$c$ = 20 N s/m", r"$c$ = 40 N s/m"]
    t = np.linspace(0, T3, int(T3 * fps * slow) + 1)
    tf = np.linspace(0, T3, 2001)
    usf = [rk4(lambda _, s, c=c: np.array([s[1], -k / m * s[0] - c / m * s[1]]), [u0, 0.0], tf)[:, 0] for c in cs]
    us = [np.interp(t, tf, u) for u in usf]

    fig = plt.figure(figsize=(9, 6))
    gs = fig.add_gridspec(2, 3, height_ratios=[1.5, 1])
    axq = fig.add_subplot(gs[1, :])
    L0, hbox = 0.06, 0.02
    objs = []
    for j in range(3):
        ax = fig.add_subplot(gs[0, j]); ax.set_aspect("equal"); ax.axis("off")
        ax.set_xlim(-0.06, 0.06); ax.set_ylim(-(L0 + u0 + hbox + 0.01), 0.01)
        hatch(ax, -0.04, 0.04, 0.0, down=False, size=0.008)
        ax.axhline(-L0, color=GREY, ls="--", lw=0.8); ax.set_title(names[j], color=cols[j], fontsize=10)
        s, = ax.plot([], [], "k", lw=1.1); d = Dashpot(ax, width=0.007)
        b = plt.Rectangle((-0.03, 0), 0.06, hbox, fc="#f4c7c3", ec="k"); ax.add_patch(b)
        objs.append((s, d, b))
        axq.plot(tf, 100 * usf[j], color=cols[j], lw=2, ls=["-", "--", ":"][j], label=names[j])
    cursor = axq.axvline(0, color="k", lw=0.8)
    axq.set_xlabel(r"$t$ (s)"); axq.set_ylabel(r"$u$ (cm)"); axq.grid(alpha=0.3); axq.legend(loc="upper right", fontsize=8)
    timer = fig.text(0.5, 0.95, "", ha="center")

    def update(i):
        for (s, d, b), u in zip(objs, us):
            y = u[i]
            s.set_data(*spring_xy((-0.012, 0), (-0.012, -(L0 + y)), width=0.006)); d.set((0.012, 0), (0.012, -(L0 + y)))
            b.set_y(-(L0 + y) - hbox)
        cursor.set_xdata([t[i], t[i]]); timer.set_text(f"$t$ = {t[i]:.2f} s   (slow motion ×{slow})")

    anim = make_anim(fig, update, len(t), fps)
    show(anim)
    return anim

anim3 = animation_3()

---
*Teacher only – export the frames of an animation for the LaTeX `animate` package* (uncomment and run):

In [ ]:
# live1.export("td2Pendulum")
# live2.export("td2MassSpringDamper")
# export_frames(anim3, "td2DampingTypes")